## Assignment: Fine-Tuning BERT for POS Tagging & Chunking

**Objective**\
Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.

**Learning Outcomes**\
After completing this assignment, students will be able to:
- Understand token classification using transformer models
- Perform POS tagging and chunking tasks
- Use BERT/DistilBERT for sequence labeling
- Handle tokenization and label alignment challenges
- Evaluate models using sequence-based metrics

**Tools and Technologies**\
Students should use the following:
- Python
- Hugging Face Transformers
- TensorFlow or PyTorch
- Jupyter Notebook / VS Code

**Task Description**\
You are required to build a token classification system using a transformer model to perform POS tagging and chunking. The task includes dataset handling, preprocessing, training, evaluation, and inference.

**Dataset Requirement**\
Choose any one dataset:
- CoNLL-2003 (for Chunking)
- Universal Dependencies (for POS Tagging)
- WikiANN (Optional)




In [ ]:
!pip install datasets transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a7b556aa0750220b5112e076a5866875a77201c1e706227aeab7c60c168fe1dd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Loading Conll2003 Dataset
from datasets import load_dataset

dataset2 = load_dataset("lhoestq/conll2003")

In [ ]:
dataset2

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
pos_tags = {'"': 0, "''": 1, '#': 2, '$': 3, '(': 4, ')': 5, ',': 6, '.': 7, ':': 8, '``': 9, 'CC': 10, 'CD': 11, 'DT': 12,
 'EX': 13, 'FW': 14, 'IN': 15, 'JJ': 16, 'JJR': 17, 'JJS': 18, 'LS': 19, 'MD': 20, 'NN': 21, 'NNP': 22, 'NNPS': 23,
 'NNS': 24, 'NN|SYM': 25, 'PDT': 26, 'POS': 27, 'PRP': 28, 'PRP$': 29, 'RB': 30, 'RBR': 31, 'RBS': 32, 'RP': 33,
 'SYM': 34, 'TO': 35, 'UH': 36, 'VB': 37, 'VBD': 38, 'VBG': 39, 'VBN': 40, 'VBP': 41, 'VBZ': 42, 'WDT': 43,
 'WP': 44, 'WP$': 45, 'WRB': 46}

chunk_tags = {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8,
 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17,
 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}

In [ ]:
id2label_pos = {v: k for k, v in pos_tags.items()}
label2id_pos = pos_tags

id2label_chunk = {v: k for k, v in chunk_tags.items()}
label2id_chunk = chunk_tags

In [ ]:
from transformers import AutoModelForTokenClassification

model_pos = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(pos_tags),
    id2label=id2label_pos,
    label2id=label2id_pos
)

model_chunk = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_tags),
    id2label=id2label_chunk,
    label2id=label2id_chunk
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Loading the BERT Tokenizer

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_chunk(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()
    labels = []
    prev = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev:
            labels.append(example["chunk_tags"][word_idx])
        else:
            labels.append(-100)
        prev = word_idx

    tokenized["labels"] = labels
    return tokenized

def tokenize_pos(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()
    labels = []
    prev = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev:
            labels.append(example["pos_tags"][word_idx])
        else:
            labels.append(-100)
        prev = word_idx

    tokenized["labels"] = labels
    return tokenized

In [ ]:
tokenized_pos = dataset2.map(tokenize_pos)
tokenized_chunk = dataset2.map(tokenize_chunk)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
tokenized_pos

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [ ]:
tokenized_chunk

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [ ]:
cols = ["id", "tokens", "pos_tags", "chunk_tags", "ner_tags", "token_type_ids"]

tokenized_pos = tokenized_pos.remove_columns(cols)
tokenized_chunk = tokenized_chunk.remove_columns(cols)

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_pos(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_pred.append(id2label_pos[int(p_)])
                temp_lab.append(id2label_pos[int(l_)])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import TrainingArguments

training_args_pos = TrainingArguments(
    output_dir="./results_pos",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer_pos = Trainer(
    model=model_pos,
    args=training_args_pos,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_pos,
)

In [ ]:
trainer_pos.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.622364,0.241398,0.913713,0.913891,0.913802
2,0.176635,0.214916,0.921769,0.920539,0.921153
3,0.131825,0.209673,0.923533,0.922480,0.923006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.25415160932953645, metrics={'train_runtime': 283.9702, 'train_samples_per_second': 148.336, 'train_steps_per_second': 9.276, 'total_flos': 510472720030266.0, 'train_loss': 0.25415160932953645, 'epoch': 3.0})

In [ ]:
pos_evaluation = trainer_pos.evaluate()

In [ ]:
trainer_pos.save_model("results_pos")
tokenizer.save_pretrained("results_pos")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('results_pos/tokenizer_config.json', 'results_pos/tokenizer.json')

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_chunk(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_pred.append(id2label_chunk[int(p_)])
                temp_lab.append(id2label_chunk[int(l_)])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import TrainingArguments

training_args_chunk = TrainingArguments(
    output_dir="./results_chunk",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer_chunk = Trainer(
    model=model_chunk,
    args=training_args_chunk,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunk,
)

In [ ]:
trainer_chunk.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.395634,0.188081,0.911115,0.904495,0.907793
2,0.146031,0.164559,0.918148,0.913082,0.915608
3,0.107878,0.161481,0.919911,0.916230,0.918067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.18728586246140003, metrics={'train_runtime': 303.805, 'train_samples_per_second': 138.651, 'train_steps_per_second': 8.67, 'total_flos': 510251380802730.0, 'train_loss': 0.18728586246140003, 'epoch': 3.0})

In [ ]:
chunk_evaluation = trainer_chunk.evaluate()

In [ ]:
trainer_chunk.save_model("results_chunk")
tokenizer.save_pretrained("results_chunk")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('results_chunk/tokenizer_config.json', 'results_chunk/tokenizer.json')

##  Inference

In [ ]:
# Loading the Trained Models

from transformers import AutoModelForTokenClassification

model_pos_loaded = AutoModelForTokenClassification.from_pretrained("results_pos")
model_chunk_loaded = AutoModelForTokenClassification.from_pretrained("results_chunk")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
sentence = "John works at Google in California"
tokens = sentence.split()

In [ ]:
inputs = tokenizer(
    tokens,
    return_tensors="pt",
    is_split_into_words=True
)

In [ ]:
import torch

with torch.no_grad():
    pos_outputs = model_pos_loaded(**inputs)
    chunk_outputs = model_chunk_loaded(**inputs)

pos_preds = torch.argmax(pos_outputs.logits, dim=2)
chunk_preds = torch.argmax(chunk_outputs.logits, dim=2)

In [ ]:
word_ids = inputs.word_ids()

In [ ]:
pos_labels_out = []
chunk_labels_out = []

for i, word_id in enumerate(word_ids):
    if word_id is not None:
        pos_label = id2label_pos[int(pos_preds[0][i])]
        chunk_label = id2label_chunk[int(chunk_preds[0][i])]

        pos_labels_out.append(pos_label)
        chunk_labels_out.append(chunk_label)

In [ ]:
print(f"Final Result\n")
for word, pos, chunk in zip(tokens, pos_labels_out, chunk_labels_out):
    print(f"{word} → POS: {pos}, CHUNK: {chunk}")

Final Result

John → POS: NNP, CHUNK: B-NP
works → POS: VBZ, CHUNK: B-VP
at → POS: IN, CHUNK: B-PP
Google → POS: NNP, CHUNK: B-NP
in → POS: IN, CHUNK: B-PP
California → POS: NNP, CHUNK: B-NP


## Model Comparison

In [ ]:
pos_results = trainer_pos.evaluate()
chunk_results = trainer_chunk.evaluate()

In [ ]:
print(f"POS Tagging Results: {pos_results}")
print(f"\nChunking Results: {chunk_results}")

POS Tagging Results: {'eval_loss': 0.20967327058315277, 'eval_precision': 0.9235328410415857, 'eval_recall': 0.9224796797282543, 'eval_f1': 0.923005959967469, 'eval_runtime': 8.4289, 'eval_samples_per_second': 385.579, 'eval_steps_per_second': 24.202, 'epoch': 3.0}

Chunking Results: {'eval_loss': 0.16148114204406738, 'eval_precision': 0.9199110556292424, 'eval_recall': 0.9162295527839297, 'eval_f1': 0.9180666134589553, 'eval_runtime': 8.4149, 'eval_samples_per_second': 386.221, 'eval_steps_per_second': 24.243, 'epoch': 3.0}


In [ ]:
pos_data = {
    "Model": "POS Tagging",
    "Precision": pos_results.get("eval_precision"),
    "Recall": pos_results.get("eval_recall"),
    "F1 Score": pos_results.get("eval_f1")
}

chunk_data = {
    "Model": "Chunking",
    "Precision": chunk_results.get("eval_precision"),
    "Recall": chunk_results.get("eval_recall"),
    "F1 Score": chunk_results.get("eval_f1")
}

In [ ]:
import pandas as pd

df_results = pd.DataFrame([pos_data, chunk_data])

In [ ]:
print(df_results)

         Model  Precision   Recall  F1 Score
0  POS Tagging   0.923533  0.92248  0.923006
1     Chunking   0.919911  0.91623  0.918067
